# Лабораторная работа 5. Большие данные и AutoML
**Вариант 9:** синтетический «финансовый» датасет (`make_classification`, 30 признаков, 5 млн строк), бинарная классификация `label`.

План: генерация → очистка и признаки в **PySpark** → разбиение 80/20 в Spark → **AutoGluon** (2 запуска с разным `time_limit`) и baseline LogReg → всё логируется в **MLflow** → финальная оценка на отложенном test.

Ядро ноутбука — отдельное окружение `.venv-lab5`: AutoGluon 1.6 требует `pandas<2.4`, а в основном `.venv` стоит pandas 3.0.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))  # чтобы импортировать src/ из notebooks/
assert Path(sys.prefix).name == ".venv-lab5", f"ядро не то: {sys.prefix}. Выберите .venv-lab5 (Select Kernel)"
import shutil
import time

import autogluon.tabular
import pandas as pd
import pyspark
import sklearn
from autogluon.tabular import TabularPredictor
from pyspark.ml.feature import Imputer
from pyspark.sql import functions as F
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from src.data import MODELS_DIR, RANDOM_STATE, ROOT
from src.lab5 import LAB5_RAW, LABEL, generate, get_spark, metrics, setup_mlflow

AG_TRAIN_ROWS = 1_000_000  # сколько строк train отдаём AutoGluon (см. раздел 4)
RUNS = {"ag_medium_5min": dict(presets="medium_quality", time_limit=300),
        "ag_medium_15min": dict(presets="medium_quality", time_limit=900)}

## 1. Данные
`generate()` (в `src/lab5.py`) создаёт 5 млн строк сразу нужного размера: 30 признаков `make_classification`, из них 12 информативных и 6 линейных комбинаций, дисбаланс 80/20, 2% случайно перевёрнутых меток.
Чтобы данные выглядели как транзакции, `feat_00` превращён в сумму `amount` (логнормальное распределение) и добавлены `client_id`, `region`, `ts`.
Специально внесены дефекты, которые Spark должен исправить: пропуски (2% в `amount`, `feat_01`, `feat_02`), 0.5% дублей строк, дата хранится строкой.
Файл `data/raw/lab5_synth.parquet` версионируется DVC.

In [ ]:
if not LAB5_RAW.exists():
    generate()  # ~1–2 мин, ~850 МБ

## 2. PySpark: загрузка и очистка
Spark работает локально (`local[*]` — все ядра). Схема берётся из Parquet, `ts` пока строка.

In [ ]:
spark = get_spark()
raw = spark.read.parquet(LAB5_RAW.as_posix())
n_raw = raw.count()
print(f"{n_raw:,} строк, {raw.rdd.getNumPartitions()} партиций")
raw.printSchema()

Очистка:
- **типы**: `ts` строка → `timestamp`, `label` и `client_id` → `int`;
- **дубли**: `dropDuplicates()` по всем колонкам (полные копии строк — ошибка выгрузки, а не новые наблюдения);
- **пропуски**: считаем по колонкам. Заполнять будем медианой **после** разбиения, по train (иначе статистика test протечёт в train). Перед заполнением сохраняем флаг `amount_missing`, потому что сам факт пропуска может быть информативен.

`cache()`: очищенный DataFrame используется дальше много раз, без кэша Spark пересчитывал бы чтение и дедупликацию на каждое действие (вычисления в Spark ленивые).

In [ ]:
df = (raw.withColumn("ts", F.to_timestamp("ts", "yyyy-MM-dd HH:mm:ss"))
         .withColumn(LABEL, F.col(LABEL).cast("int"))
         .withColumn("client_id", F.col("client_id").cast("int"))
         .dropDuplicates()
         .cache())
n_clean = df.count()
print(f"дублей удалено: {n_raw - n_clean:,} → {n_clean:,} строк")

nulls = df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns]).toPandas().iloc[0]
NULL_COLS = nulls[nulls > 0].index.tolist()
nulls[NULL_COLS]

## 3. Разбиение 80/20 и признаки
Разбиваем **по хешу строки** (`xxhash64` всех колонок, остаток от деления на 10): строка всегда попадает в одну и ту же часть.
`randomSplit` зависит от порядка строк в партициях, а после `dropDuplicates` (shuffle) этот порядок между запусками не гарантирован, поэтому `randomSplit` был бы невоспроизводим.
При 5 млн строк стратификация не нужна: доли классов в train и test совпадают и так (проверка ниже).

In [ ]:
bucket = F.pmod(F.xxhash64(*df.columns), 10)
train, test = df.where(bucket < 8), df.where(bucket >= 8)
pd.DataFrame({name: part.groupBy(LABEL).count().toPandas().set_index(LABEL)["count"]
              for name, part in [("train", train), ("test", test)]}).pipe(lambda t: t / t.sum()).round(4)

**Импутация:** `pyspark.ml.feature.Imputer` (медиана) обучается только на train и применяется к обеим частям — то же, что `SimpleImputer` внутри Pipeline в прошлых лабах.

**Новые признаки** (11 шт., требование — не менее 3):
| Группа | Признаки | Зачем |
|---|---|---|
| преобразование | `log_amount = log1p(amount)` | суммы логнормальные; для линейной модели лог делает связь линейной |
| дата | `hour`, `day_of_week`, `month`, `is_weekend` | из `ts` извлекаем то, с чем умеет работать модель |
| групповые статистики по клиенту | `client_txn_cnt`, `client_amount_mean`, `client_amount_std`, `amount_to_client_mean` | насколько операция типична для клиента — классика антифрода |
| групповая по региону | `region_amount_mean` | средний чек региона |
| флаг | `amount_missing` | факт пропуска |

Групповые статистики считаются **только по train** и присоединяются к обеим частям через `join`. Если клиента нет в train, ставим количество 0, std 0 и средний чек региона. Std 0 ставим и клиенту с одной операцией: для него std не определено (`broadcast` — таблица клиентов маленькая, 100 тыс. строк, её выгоднее разослать на все исполнители, чем делать shuffle 5 млн строк).
После извлечения признаков убираем `ts` и `client_id`: идентификатор как число модели бессмыслен, а информацию о клиенте уже несут агрегаты.

In [ ]:
imputer = Imputer(strategy="median", inputCols=NULL_COLS, outputCols=NULL_COLS)

def row_features(sdf):
    return (sdf.withColumn("amount_missing", F.col("amount").isNull().cast("int"))
               .withColumn("hour", F.hour("ts"))
               .withColumn("day_of_week", F.dayofweek("ts"))
               .withColumn("month", F.month("ts"))
               .withColumn("is_weekend", F.dayofweek("ts").isin(1, 7).cast("int")))

train, test = row_features(train), row_features(test)
imputer = imputer.fit(train)
train, test = imputer.transform(train), imputer.transform(test)

client_stats = train.groupBy("client_id").agg(F.count("*").alias("client_txn_cnt"),
                                              F.mean("amount").alias("client_amount_mean"),
                                              F.stddev("amount").alias("client_amount_std"))
region_stats = train.groupBy("region").agg(F.mean("amount").alias("region_amount_mean"))

def group_features(sdf):
    return (sdf.join(F.broadcast(client_stats), "client_id", "left")
               .join(F.broadcast(region_stats), "region", "left")
               # клиент не встречался в train (или у него одна операция, std не определено) — нейтральные значения
               .fillna({"client_txn_cnt": 0, "client_amount_std": 0.0})
               .withColumn("client_amount_mean", F.coalesce("client_amount_mean", "region_amount_mean"))
               .withColumn("log_amount", F.log1p("amount"))
               .withColumn("amount_to_client_mean", F.col("amount") / F.col("client_amount_mean"))
               .drop("ts", "client_id"))

train, test = group_features(train), group_features(test)
train.printSchema()

In [ ]:
head = train.limit(10).toPandas()
(ROOT / "reports").mkdir(exist_ok=True)
(ROOT / "reports" / "lab5_schema.txt").write_text(
    "\n".join(f"{f.name}: {f.dataType.simpleString()}" for f in train.schema), encoding="utf-8")
head.to_csv(ROOT / "reports" / "lab5_head.csv", index=False)
head

## 4. Выгрузка в pandas
AutoGluon работает с pandas, поэтому данные выгружаем через `toPandas()` (с Arrow — колоночная передача без построчной сериализации).
Train после разбиения — около 4 млн строк. AutoGluon с лимитом 5–15 минут на них успел бы обучить одну-две модели, поэтому отдаём ему **случайную подвыборку 1 млн строк** (методичка это разрешает). Test выгружаем целиком (~1 млн): оценка на всём отложенном test.

In [ ]:
n_train = train.count()
train_pdf = train.sample(fraction=min(1.0, AG_TRAIN_ROWS / n_train), seed=RANDOM_STATE).toPandas()
test_pdf = test.toPandas()
X_test, y_test = test_pdf.drop(columns=LABEL), test_pdf[LABEL]
assert not (train_pdf.isna().any().any() or test_pdf.isna().any().any()), "после Spark остались пропуски"
print(f"train в Spark: {n_train:,}, подвыборка для AutoML: {len(train_pdf):,}, test: {len(test_pdf):,}, "
      f"признаков: {X_test.shape[1]}, RAM: {(train_pdf.memory_usage().sum() + test_pdf.memory_usage().sum()) / 2**20:.0f} МБ")

## 5. MLflow и baseline
MLflow локально: метаданные запусков — в `mlflow.db` (SQLite), артефакты (модели, leaderboard) — в `mlruns/`. Эксперимент назван по варианту.
В каждый запуск пишем одинаковый набор параметров (размеры данных, версии библиотек), чтобы запуски сравнивались в UI.

**Baseline** — логистическая регрессия: StandardScaler для чисел, OneHot для `region`. Обучаем на той же подвыборке.

In [ ]:
mlflow = setup_mlflow()
common = {"n_rows_raw": n_raw, "n_rows_clean": n_clean, "n_train_spark": n_train, "n_train_used": len(train_pdf),
          "n_test": len(test_pdf), "n_features": X_test.shape[1], "python": sys.version.split()[0],
          "pyspark": pyspark.__version__, "autogluon": autogluon.tabular.__version__, "sklearn": sklearn.__version__}
results = {}

def log_results(name):
    mlflow.log_metrics({k if k == "fit_time_s" else f"test_{k}": v for k, v in results[name].items()})

num_cols = [c for c in X_test.columns if c != "region"]
baseline = make_pipeline(
    ColumnTransformer([("num", StandardScaler(), num_cols), ("cat", OneHotEncoder(handle_unknown="ignore"), ["region"])]),
    LogisticRegression(max_iter=1000))
with mlflow.start_run(run_name="baseline_logreg"):
    t0 = time.time()
    baseline.fit(train_pdf.drop(columns=LABEL), train_pdf[LABEL])
    results["baseline_logreg"] = metrics(y_test, baseline.predict_proba(X_test)[:, 1]) | {"fit_time_s": time.time() - t0}
    mlflow.log_params(common | {"library": "sklearn", "model": "LogisticRegression"})
    log_results("baseline_logreg")
    mlflow.sklearn.log_model(baseline, name="model")
results["baseline_logreg"]

## 6. AutoGluon
`TabularPredictor` сам определяет типы признаков, сам отделяет валидационную часть от train и по ней выбирает модели и строит `WeightedEnsemble` (взвешенное среднее лучших). Метрика — `roc_auc`.

Два запуска с `presets="medium_quality"` (быстрый пресет без бэггинга, рекомендован методичкой) и разным `time_limit`: 5 и 15 минут.
Исключены `KNN`, `RF`, `XT`: на 1 млн строк они занимают гигабайты памяти и диска и медленно предсказывают, а для табличных данных сильнее бустинги. `NN_TORCH` и `FASTAI` пропускаются, потому что torch не установлен (экономим ~2 ГБ).

Выбор лучшей модели — по `score_val` (валидация внутри train). Test в выборе не участвует.
В MLflow логируем параметры AutoML, метрики, leaderboard и саму модель: `clone_for_deployment` — копия только с лучшей моделью и тем, что нужно для предсказания.

In [ ]:
EXCLUDED = ["KNN", "RF", "XT", "NN_TORCH", "FASTAI"]
leaderboards, deploy_paths = {}, {}
for name, cfg in RUNS.items():
    with mlflow.start_run(run_name=name):
        t0 = time.time()
        pred = TabularPredictor(LABEL, eval_metric="roc_auc", path=str(ROOT / "AutogluonModels" / name), verbosity=1)
        pred.fit(train_pdf, excluded_model_types=EXCLUDED, **cfg)
        lb = leaderboards[name] = pred.leaderboard()
        results[name] = metrics(y_test, pred.predict_proba(X_test)[1].to_numpy()) | {"fit_time_s": time.time() - t0}

        mlflow.log_params(common | cfg | {"library": "autogluon", "excluded": ",".join(EXCLUDED),
                                          "best_model": pred.model_best, "n_models": len(lb)})
        mlflow.log_metric("val_roc_auc", lb["score_val"].max())
        log_results(name)
        mlflow.log_text(lb.to_csv(index=False), "leaderboard.csv")
        deploy_paths[name] = pred.clone_for_deployment(str(ROOT / "AutogluonModels" / f"{name}_deploy"))
        mlflow.log_artifacts(deploy_paths[name], "model")
    print(f"{name}: моделей в leaderboard {len(lb)}, лучшая {pred.model_best}, val ROC-AUC {lb['score_val'].max():.4f}")

In [ ]:
for name, lb in leaderboards.items():
    print(name)
    display(lb[["model", "score_val", "fit_time", "pred_time_val", "stack_level"]].round(4))

## 7. Сравнение на отложенном test
Test (≈1 млн строк) не видели ни AutoGluon, ни baseline. При таком объёме 95% доверительный интервал ROC-AUC порядка ±0.001, так что различия в третьем знаке уже значимы — в отличие от Titanic в прошлых лабах.

In [ ]:
summary = pd.DataFrame(results).T.round(4)
summary

In [ ]:
# лучший запуск выбираем по валидации, а не по test; его модель сохраняем в models/ под DVC
best_run = max(leaderboards, key=lambda n: leaderboards[n]["score_val"].max())
shutil.copytree(deploy_paths[best_run], MODELS_DIR / "lab5_autogluon", dirs_exist_ok=True)
print("лучший запуск:", best_run, "→ models/lab5_autogluon")

mlflow.search_runs(order_by=["start_time DESC"])[
    ["tags.mlflow.runName", "params.library", "params.time_limit", "params.best_model",
     "metrics.val_roc_auc", "metrics.test_roc_auc", "metrics.test_f1", "metrics.fit_time_s"]].round(4)

**MLflow UI** — из корня проекта, в основном окне PowerShell с активированным `.venv-lab5`:
```
mlflow ui --backend-store-uri sqlite:///mlflow.db
```
Открыть http://127.0.0.1:5000, эксперимент `lab5_variant9_synthetic_financial`, отметить три запуска → **Compare**. Скриншоты — в `reports/`.

In [ ]:
spark.stop()

## Выводы
Итоговые цифры — в таблице раздела 7 и в README.
- **PySpark.** 5 млн строк × 34 колонки (~850 МБ на диске) pandas ещё переварил бы на одной машине, но Spark даёт то же решение, которое масштабируется на кластер без переписывания: ленивые вычисления, `cache`, `broadcast`-join, встроенные функции без Python UDF (все вычисления остаются в JVM). Цена — медленный старт JVM и накладные расходы, на маленьких данных Spark проигрывает pandas.
- **AutoML.** За минуты без ручной настройки получен ансамбль бустингов; leaderboard сразу показывает, какие семейства моделей работают. Ограничения: модель — чёрный ящик из нескольких моделей, тяжелее в деплое (поэтому `clone_for_deployment`), и прирост от увеличения времени убывает.
- **Baseline.** Логистическая регрессия сильно проигрывает. `make_classification` кладёт каждый класс в несколько гауссовых кластеров (`n_clusters_per_class=2`) в вершинах гиперкуба, поэтому граница между классами нелинейная, а деревья её находят. Здесь AutoML оправдан: ручной LogReg на порядок хуже по PR-AUC и F1.
- **MLflow** хранит параметры, версии библиотек, размеры данных и модели каждого запуска, поэтому любой результат можно воспроизвести и сравнить.